# Soybean Strategy Backtest Research

Standalone research notebook for soybean. The strategy logic is inside this notebook: generic A/B tests, price Momentum/MR, annual walk-forward, Cargill overlay, and named-period check.

In [25]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "support"))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from shared_backtest import (
    active_metrics,
    backtest_signal,
    build_feature_panels,
    clean_signal,
    load_train_set,
    metric_row as shared_metric_row,
    named_period_check as shared_named_period_check,
    run_family_tests,
)
from research_config import SPLIT_DATE

DATA_DIR = "support/train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
TRAIN_END = pd.Timestamp("2016-01-01")
DD_CAPITAL_USD = 10_000.0

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

COMMODITY = "SOYABEAN"


## 1. Load Data And Confirm Cargill Inputs

In [26]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_feature_panels(data)
trading_index = futures_pnl_all.index
futures_pnl = futures_pnl_all[[COMMODITY]]
panel = feature_panels[COMMODITY].reindex(trading_index).fillna(0.0)

display(pd.DataFrame([{
    "commodity": COMMODITY,
    "rows": len(trading_index),
    "start": trading_index.min(),
    "end": trading_index.max(),
    "has_cargill_inputs": {"cgl_inventory_change", "crush_surprise", "crush_utilization"}.issubset(panel.columns),
    "train_rows": int((trading_index < TRAIN_END).sum()),
    "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
    "oos_rows": int((trading_index >= OOS_START).sum()),
}]))

,commodity,rows,start,end,has_cargill_inputs,train_rows,validation_rows,oos_rows
0,SOYABEAN,2866,2010-01-04,2020-12-31,True,1565,520,781


## 2. Standalone Helpers

In [27]:
def metric_row(label, bt):
    return shared_metric_row(label, bt, TRAIN_END, OOS_START, DD_CAPITAL_USD)


def named_period_check(bt):
    return shared_named_period_check(bt, DD_CAPITAL_USD)


## 3. Generic Signal A/B Tests

In [28]:
families_A = {
    "price_trend": [panel["mom_20"], panel["mom_60"]],
    "price_mr": [panel["rev_5"]],
    "curve": [panel["curve_spread"], panel["curve_ratio"], panel["curve_change_20"]],
    "cot": [panel["cot_mm_level"], panel["cot_mm_change"], panel["cot_pm_oi_level"], panel["cot_pm_oi_change"]],
}
families_B = {
    **families_A,
    "physical_public": [-panel["public_inventory_change"], -panel["receipts_change"]],
    "physical_cargill": [-panel["cgl_inventory_change"], panel["crush_surprise"], panel["crush_utilization"]],
}


def generic_metric_row(signal_set, name, bt):
    row = {"signal_set": signal_set}
    row.update(metric_row(name, bt))
    return row


generic = run_family_tests(
    {"A": families_A, "B": families_B},
    futures_pnl[COMMODITY].shift(-1),
    trading_index,
    lambda signal: backtest_signal(signal, futures_pnl, COMMODITY),
    generic_metric_row,
    trend_panel=panel,
)
generic_results = generic["results"]
display(generic_results.round(3))


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.180,0.758,0.758,"1,305.316",-11.056,0.480,0.006
3,A,select_by_ic,long_short,0.191,0.718,0.342,365.363,-7.606,0.378,0.004
5,A,kalman_family_model,long_short,-0.027,0.683,0.294,500.413,-14.819,0.250,0.006
4,A,expanding_ols_family_model,long_short,-0.693,0.637,0.282,442.340,-13.131,-0.012,0.006
1,A,equal_family,long_short,0.058,0.577,0.546,416.848,-7.438,0.306,0.002
0,A,avg_all_signals,long_short,0.078,0.486,0.635,511.507,-8.331,0.333,0.002
10,B,expanding_ols_family_model,long_short,-0.782,1.142,0.486,572.316,-5.800,0.153,0.006
8,B,best_family_by_trend_mr,long_short,0.180,0.758,0.758,"1,305.316",-11.056,0.480,0.006
11,B,kalman_family_model,long_short,-0.064,0.742,0.351,499.323,-8.386,0.267,0.006
6,B,avg_all_signals,long_short,-0.062,0.015,0.393,205.565,-4.164,0.081,0.002


## 4. Momentum/MR Price Benchmark

In [29]:
trend_gate = panel["mom_60"].abs() > panel["mom_60"].abs().expanding(min_periods=252).median().shift(1)
price_signals = {
    "mom20": clean_signal(panel["mom_20"], trading_index),
    "mom60": clean_signal(panel["mom_60"], trading_index),
    "mr5": clean_signal(panel["rev_5"], trading_index),
    "mom60_mr5_equal": clean_signal((panel["mom_60"] + panel["rev_5"]) / 2.0, trading_index),
    "trend_mom_else_mr": clean_signal(panel["mom_60"].where(trend_gate, panel["rev_5"]), trading_index),
}
price_rows = [metric_row(name, backtest_signal(signal, futures_pnl, COMMODITY)) for name, signal in price_signals.items()]
display(pd.DataFrame(price_rows).sort_values("oos_sharpe", ascending=False).round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
1,mom60,long_short,0.465,0.666,0.981,"1,928.351",-12.034,0.665,0.003
4,trend_mom_else_mr,long_short,0.223,0.799,0.914,"1,849.798",-12.689,0.561,0.006
3,mom60_mr5_equal,long_short,0.211,0.401,0.807,859.294,-6.935,0.422,0.004
0,mom20,long_short,0.179,0.440,0.389,633.967,-10.798,0.293,0.005
2,mr5,long_short,-0.341,-0.302,-0.527,-616.086,-10.208,-0.386,0.008


## 5. Annual Walk-Forward Momentum/MR

In [30]:
price_backtests = {name: backtest_signal(signal, futures_pnl, COMMODITY) for name, signal in price_signals.items()}
walk_forward_signal = pd.Series(0.0, index=trading_index)
wf_rows = []
for year in range(OOS_START.year, trading_index.max().year + 1):
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year + 1}-01-01")
    validation_start = start - pd.DateOffset(years=2)
    train_mask = trading_index < validation_start
    validation_mask = (trading_index >= validation_start) & (trading_index < start)
    trade_mask = (trading_index >= start) & (trading_index < end)
    scored = []
    for name, bt in price_backtests.items():
        train = active_metrics(bt, train_mask)
        validation = active_metrics(bt, validation_mask)
        scored.append({"year": year, "candidate": name, "train_sharpe": train["sharpe"], "validation_sharpe": validation["sharpe"], "score": validation["sharpe"] if train["sharpe"] > 0 else -np.inf})
    score_table = pd.DataFrame(scored)
    selected = score_table.sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
    walk_forward_signal.loc[trade_mask] = price_signals[selected].loc[trade_mask]
    trade = active_metrics(price_backtests[selected], trade_mask)
    wf_rows.append({"year": year, "selected": selected, "trade_sharpe": trade["sharpe"], "trade_pnl": trade["total_pnl"], "trade_dd_pct": trade["max_drawdown"] / DD_CAPITAL_USD * 100.0, "active_days": trade["days"]})

walk_forward_bt = backtest_signal(walk_forward_signal, futures_pnl, COMMODITY)
display(pd.DataFrame(wf_rows).round(3))
display(pd.DataFrame([metric_row("annual_walk_forward_momentum_mr", walk_forward_bt)]).round(3))

,year,selected,trade_sharpe,trade_pnl,trade_dd_pct,active_days
0,2018,trend_mom_else_mr,0.823,603.684,-5.222,252.000
1,2019,mom60,-2.037,-738.549,-10.218,229.000
2,2020,mom20,0.833,496.796,-5.153,242.000


,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
0,annual_walk_forward_momentum_mr,long_short,NaN,NaN,0.251,433.948,-13.938,0.251,0.006


## 6. Cargill Overlay And Final Candidate

In [31]:
cargill_physical = clean_signal((-panel["cgl_inventory_change"] + panel["crush_surprise"] + panel["crush_utilization"]) / 3.0, trading_index)
disagreement = np.sign(walk_forward_signal) * np.sign(cargill_physical) < 0.0
final_signal = clean_signal(walk_forward_signal.where(~disagreement, 0.50 * walk_forward_signal), trading_index)

pnl_vol = futures_pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1)
low_vol = pnl_vol < (0.80 * pnl_vol.expanding(min_periods=252).median().shift(1))
final_signal = clean_signal(final_signal.where(~low_vol.reindex(trading_index).fillna(False), 0.0), trading_index)

final_bt = backtest_signal(final_signal, futures_pnl, COMMODITY)
final_row = pd.DataFrame([metric_row("wf_momentum_mr + cargill_disagree_half + low_vol_flat", final_bt)])
display(final_row.round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
0,wf_momentum_mr + cargill_disagree_half + low_v...,long_short,NaN,NaN,1.893,668.552,-3.503,1.893,0.005


## 7. Named-period Check

In [32]:
period_check = named_period_check(final_bt)
display(period_check.round(3))
row = final_row.iloc[0]
display(Markdown(f"""
## Conclusion

Final soybean candidate: `wf_momentum_mr + cargill_disagree_half + low_vol_flat`.

OOS Sharpe `{row['oos_sharpe']:.3f}`, OOS PnL `{row['oos_pnl']:.3f}`, OOS max DD `{row['oos_dd_pct']:.3f}%`.

The generic A/B tests are kept as diagnostics. The promoted rule is the walk-forward Momentum/MR spine with Cargill as a physical disagreement haircut, not a fitted coefficient model.
"""))

,period,total_pnl,sharpe,max_dd_pct,hit_rate,active_days,note
0,2010-2011: Russian drought/export ban shock,NaN,NaN,NaN,NaN,0,before first active trade (2018)
1,2012-2013: US drought rally/retrace,NaN,NaN,NaN,NaN,0,before first active trade (2018)
2,2014: Crimea/Black Sea shock,NaN,NaN,NaN,NaN,0,before first active trade (2018)
3,2014-2017: Low-price abundant supply,NaN,NaN,NaN,NaN,0,before first active trade (2018)
4,2018-2020: US-China trade war,-144.082,-0.703,-3.503,0.484,122,
5,2019: 2019 prevented planting floods,-84.164,-6.405,-0.839,0.308,13,
6,2020: COVID demand shock,NaN,NaN,NaN,NaN,0,strategy flat in this period
7,2020: COVID recovery/China buying,173.890,2.299,-1.471,0.635,52,



## Conclusion

Final soybean candidate: `wf_momentum_mr + cargill_disagree_half + low_vol_flat`.

OOS Sharpe `1.893`, OOS PnL `668.552`, OOS max DD `-3.503%`.

The generic A/B tests are kept as diagnostics. The promoted rule is the walk-forward Momentum/MR spine with Cargill as a physical disagreement haircut, not a fitted coefficient model.
